# RAG to Agent: Foundation

Builds from a basic RAG pipeline to a fully agentic system, showing each step manually.

## Concepts covered
- **RAG pipeline**: read docs → chunk → index → search → build prompt → call LLM
- **Tool/function calling**: how to define a tool as JSON, and what the LLM returns when it "calls" one
- **Step-by-step tool call flow**: the 6 steps from LLM request to final answer
- **Structured output**: using Pydantic to enforce a specific response shape

In [1]:
from openai import OpenAI
openai_client = OpenAI()

## Part 1: Basic RAG Pipeline

Classic RAG: search for relevant docs, inject them into a prompt, call the LLM once.
The LLM has no tools — it only sees what we put in the prompt.

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [4]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [5]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.
Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [6]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [7]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

In [8]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [ ]:
print(response.output_text)

## Making it Agentic

In [9]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: if you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function", 
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [10]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

62

In [ ]:
# STEP 1: Ask the model, and describe what tools it can use.
# Adding search_tool here does NOT run a search.
# It just tells the model "you have a search function available if you need it."

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

response.usage.input_tokens

In [22]:
response

Response(id='resp_0b461828b9850e970069f4e49def00819588bee60528b724cb', created_at=1777656989.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"create a dashboard in Evidently"}', call_id='call_pltERV0XbXHyZhSINNMBqSsP', name='search', type='function_call', id='fc_0b461828b9850e970069f4e49e7908819585e540b51935c801', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'The search query to look up in the index'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the documentation database for relevant results based on a query string.')], top_p=1.0, background=False, completed_at=1777656990.0, conversation=None, max_outpu

In [ ]:
# STEP 2: The model didn't answer the question — it returned a tool call request instead.
# tool_call is NOT the search result. It's the model saying:
# "I need to search, please run search('create a dashboard in Evidently') for me."
# The model can't run code itself — it can only ask us to.
tool_call = response.output[0]

In [13]:
tool_call

ResponseFunctionToolCall(arguments='{"query":"create a dashboard in Evidently"}', call_id='call_pltERV0XbXHyZhSINNMBqSsP', name='search', type='function_call', id='fc_0b461828b9850e970069f4e49e7908819585e540b51935c801', namespace=None, status='completed')

In [ ]:
# STEP 3: Add the model's tool request to the conversation history.
# We need this so the next API call knows what the model previously asked for.
messages.append(tool_call)

In [16]:
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant.\n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: if you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create a dashboard in Evidently"}', call_id='call_pltERV0XbXHyZhSINNMBqSsP', name='search', type='function_call', id='fc_0b461828b9850e970069f4e49e7908819585e540b51935c801', namespace=None, status='completed')]

In [17]:
tool_call.arguments

'{"query":"create a dashboard in Evidently"}'

In [ ]:
# STEP 4: WE run the search ourselves, using the arguments the model requested.
# The model's arguments come back as a JSON string, so we parse it first.
arguments = json.loads(tool_call.arguments)

In [21]:
arguments

{'query': 'create a dashboard in Evidently'}

In [19]:
search_results = search(query='create dashboard in Evidently')

In [ ]:
# This is where the actual search happens — our Python code, not the model.
# **arguments unpacks {'query': 'create a dashboard in Evidently'} as a keyword argument.
search_results = search(**arguments)
search_results[:]

In [ ]:
# STEP 5: Package the search results in the format OpenAI expects.
# call_id links this result back to the specific tool_call request from Step 2,
# so the model knows which search result matches which request it made.
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [ ]:
# Add the search results to the conversation history.
# messages now contains: system → user question → model's tool request → search results
messages.append(call_output)
messages

In [ ]:
# STEP 6: Call the model again with the full conversation history.
# Now the model has everything it needs: the question, what it searched for, and the results.
# This time it should return an actual text answer instead of another tool call.
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

response.usage.input_tokens

In [ ]:
print(response.output_text)

## Structured Output

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [ ]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

In [ ]:
rag_response=response.output_parsed
print(rag_response.answer)